# Geometric Stiffness Matrix Method with Incremental Load Application

## Objective
To calculate nodal displacements, member end forces, and support
reactions using geometric stiffness and incremental load application.

## Method
1. Import the existing FEM model and elastic solution.
2. Divide the total load into user-defined load steps.
3. Estimate axial forces in the current configuration.
4. Assemble the element and global geometric stiffness matrices.
5. Apply Newton-type iterative corrections subject to support constraints.
6. Check convergence at each load step.
7. Report the final displacement and member-force results.

In [1]:
import pickle
from pathlib import Path
import numpy as np

DATA_FILE = Path("geometric_start.pkl")
if not DATA_FILE.exists():
    raise FileNotFoundError(
        "geometric_start.pkl not found. Open BTP(7).ipynb, run through Cell 50 "
        "after analyze_structure() succeeds, and copy geometric_start.pkl into "
        "this notebook's folder. Only load pickle files you trust."
    )

with DATA_FILE.open("rb") as f:
    data = pickle.load(f)

FEM_MODEL = data["FEM_MODEL"]
ETYPE = str(data["ETYPE"]).upper()
K_ELASTIC = np.asarray(data["K_GLOBAL"], dtype=float)
F_FULL = np.asarray(data["F_GLOBAL"], dtype=float).reshape(-1)
C = np.asarray(data["C_CONSTRAINT"], dtype=float)
ELEMENT_RESULTS = data["ELEMENT_RESULTS"]
ELEMENT_LOAD_VECTORS = data["ELEMENT_LOAD_VECTORS"]
NODE_IDS = data["NODE_IDS"]
DOF_PER_NODE = int(data["DOF_PER_NODE"])
TOTAL_DOF = int(data["TOTAL_DOF"])

E = float(FEM_MODEL["E"])
A = float(FEM_MODEL["A"])
I = FEM_MODEL.get("I")

if ETYPE not in {"TRUSS", "BEAM", "FRAME"}:
    raise ValueError(f"Unsupported element type: {ETYPE}")
if K_ELASTIC.shape != (TOTAL_DOF, TOTAL_DOF):
    raise ValueError(f"K_ELASTIC shape {K_ELASTIC.shape} != ({TOTAL_DOF}, {TOTAL_DOF})")
if F_FULL.shape != (TOTAL_DOF,):
    raise ValueError("F_GLOBAL shape does not match TOTAL_DOF")
if C.ndim != 2 or C.shape[1] != TOTAL_DOF:
    raise ValueError("Constraint matrix shape does not match TOTAL_DOF")
if not np.allclose(K_ELASTIC, K_ELASTIC.T, atol=1e-7, rtol=1e-8):
    raise ValueError("Exported elastic global stiffness matrix is not symmetric")
if E <= 0 or A <= 0:
    raise ValueError("E and A must be positive")
if ETYPE in {"BEAM", "FRAME"} and (I is None or float(I) <= 0):
    raise ValueError("Beam/frame analysis requires positive I")

print("Loaded exported model")
print("Type:", ETYPE)
print("Segments:", len(ELEMENT_RESULTS))
print("Total DOFs:", TOTAL_DOF)
print("K matrix:", K_ELASTIC.shape)
print("Constraints:", C.shape)

Loaded exported model
Type: TRUSS
Segments: 6
Total DOFs: 8
K matrix: (8, 8)
Constraints: (5, 8)


In [2]:
# ============================================================
# MATERIAL PROPERTY AND UNIT VALIDATION
# ============================================================

E = float(FEM_MODEL["E"])
A = float(FEM_MODEL["A"])
I = FEM_MODEL.get("I")

# This notebook uses SI units:
# Length       : m
# Force        : N
# Stress       : Pa
# Area         : m²
# Second moment: m⁴

if E <= 0:
    raise ValueError("Young's modulus must be positive.")

if A <= 0:
    raise ValueError("Cross-sectional area must be positive.")

if ETYPE in ("BEAM", "FRAME"):

    if I is None or float(I) <= 0:
        raise ValueError("Positive second moment of area required.")

    I = float(I)

print("\nMATERIAL PROPERTY CHECK")
print("-" * 45)

print(f"Element type : {ETYPE}")
print(f"E            : {E:.6e} Pa")
print(f"A            : {A:.6e} m²")
print(f"EA           : {E*A:.6e} N")

if I is not None:
    print(f"I            : {I:.6e} m⁴")
    print(f"EI           : {E*I:.6e} N.m²")

print("-" * 45)


MATERIAL PROPERTY CHECK
---------------------------------------------
Element type : TRUSS
E            : 1.000000e+11 Pa
A            : 1.000000e-04 m²
EA           : 1.000000e+07 N
---------------------------------------------


In [3]:
n_steps = int(input("Number of load steps (e.g. 2, 5, 10): "))
if n_steps < 1:
    raise ValueError("Number of load steps must be at least 1")

# This is the variable load increment factor requested:
# 2 steps -> 0.5; 5 steps -> 0.2; 10 steps -> 0.1.
load_increment = 1.0 / n_steps
max_iterations = 50
displacement_tolerance = 1e-8
residual_tolerance = 1e-7
relaxation = 1.0  # Optional update damping, not the load increment.

print(f"Load increment = 1/{n_steps} = {load_increment:.6f}")
print(f"Max iterations per step = {max_iterations}")
print(f"Displacement tolerance = {displacement_tolerance:.2e}")
print(f"Residual tolerance = {residual_tolerance:.2e}")

Number of load steps (e.g. 2, 5, 10):  


ValueError: invalid literal for int() with base 10: ''

In [64]:
def constraint_nullspace(Cmat, ndof):
    """Return Z with Cmat @ Z approximately zero, so admissible u = Z @ q."""
    Cmat = np.asarray(Cmat, dtype=float)
    if Cmat.ndim != 2 or Cmat.shape[1] != ndof:
        raise ValueError("Invalid constraint matrix dimensions")
    if Cmat.shape[0] == 0:
        raise ValueError("No support constraints found")
    _, singular_values, Vt = np.linalg.svd(Cmat, full_matrices=True)
    if singular_values.size:
        tol = max(Cmat.shape) * np.finfo(float).eps * singular_values[0]
        rank = int(np.sum(singular_values > tol))
    else:
        rank = 0
    Z = Vt[rank:].T
    if not np.allclose(Cmat @ Z, 0.0, atol=1e-8):
        raise RuntimeError("Failed to construct constraint null space")
    return Z

Z = constraint_nullspace(C, TOTAL_DOF)
if Z.shape[1] == 0:
    raise ValueError("All degrees of freedom are constrained; no solve is required")
print("Free generalized DOFs:", Z.shape[1])
print("Constraint-nullspace check:", np.linalg.norm(C @ Z))

def solve_constrained(K, F):
    """Solve K u = F subject to C u = 0 using a null-space basis."""
    Kr = Z.T @ K @ Z
    Fr = Z.T @ F
    if not np.allclose(Kr, Kr.T, atol=1e-7, rtol=1e-8):
        raise RuntimeError("Reduced tangent matrix is not symmetric")
    if np.linalg.matrix_rank(Kr) < Kr.shape[0]:
        raise np.linalg.LinAlgError("Reduced tangent matrix is singular/unstable")
    q = np.linalg.solve(Kr, Fr)
    return Z @ q, Kr

Free generalized DOFs: 3
Constraint-nullspace check: 9.957992501029599e-17


In [65]:
# ============================================================
# CELL 4: ELEMENT DISPLACEMENT AND AXIAL FORCE
# ============================================================

def local_displacement(result, u):

    dofs = np.asarray(result["dofs"], dtype=int)
    T = np.asarray(result["T"], dtype=float)

    ug = np.asarray(u, dtype=float)[dofs]
    ul = T @ ug

    return dofs, T, ug, ul


def axial_force_from_state(result, u):
    """
    Calculate axial force using local axial end displacements.

    Positive = Tension
    Negative = Compression

    This is the small-displacement axial-force formulation.
    """

    _, _, _, ul = local_displacement(result, u)

    L = float(result["L"])

    if L <= 1e-12:
        raise ValueError("Zero-length element detected.")

    if ETYPE == "TRUSS":

        if len(ul) != 4:
            raise ValueError("Invalid truss DOF layout.")

        axial_i = 0
        axial_j = 2

    elif ETYPE in ("BEAM", "FRAME"):

        if len(ul) != 6:
            raise ValueError("Invalid beam/frame DOF layout.")

        axial_i = 0
        axial_j = 3

    else:

        raise ValueError(f"Unsupported element type: {ETYPE}")

    delta = ul[axial_j] - ul[axial_i]

    N = (E * A / L) * delta

    return N


# ------------------------------------------------------------
# ELEMENT METADATA VALIDATION
# ------------------------------------------------------------

for sid, result in ELEMENT_RESULTS.items():

    expected = 4 if ETYPE == "TRUSS" else 6

    if np.asarray(result["k_local"]).shape != (expected, expected):
        raise ValueError(f"Segment {sid}: invalid local stiffness matrix.")

    if np.asarray(result["T"]).shape != (expected, expected):
        raise ValueError(f"Segment {sid}: invalid transformation matrix.")

    if np.asarray(result["dofs"]).size != expected:
        raise ValueError(f"Segment {sid}: invalid DOF mapping.")

    if float(result["L"]) <= 1e-12:
        raise ValueError(f"Segment {sid}: invalid element length.")

print("Element displacement and axial force extraction validated.")

Element displacement and axial force extraction validated.


In [66]:
# ============================================================
# GENERAL ELEMENT GEOMETRIC STIFFNESS
# ============================================================

def geometric_stiffness_local(N, L, etype):

    etype = etype.upper()

    L = float(L)
    N = float(N)

    if L <= 1e-12:
        raise ValueError("Element length must be positive.")

    # --------------------------------------------------------
    # 2D TRUSS
    # Local DOFs: ui, vi, uj, vj
    # --------------------------------------------------------

    if etype == "TRUSS":

        KG = (N / L) * np.array([

            [0,  0,  0,  0],
            [0,  1,  0, -1],
            [0,  0,  0,  0],
            [0, -1,  0,  1]

        ], dtype=float)

        return KG

    # --------------------------------------------------------
    # 2D BEAM / FRAME
    # Local DOFs: ui, vi, ri, uj, vj, rj
    # --------------------------------------------------------

    if etype in ("BEAM", "FRAME"):

        L2 = L**2

        KG = (N / (30.0 * L)) * np.array([

            [0, 0, 0, 0, 0, 0],

            [0, 36, 3*L, 0, -36, 3*L],

            [0, 3*L, 4*L2, 0, -3*L, -L2],

            [0, 0, 0, 0, 0, 0],

            [0, -36, -3*L, 0, 36, -3*L],

            [0, 3*L, -L2, 0, -3*L, 4*L2]

        ], dtype=float)

        return KG

    raise ValueError(f"Unsupported element type: {etype}")


# ------------------------------------------------------------
# MATRIX VALIDATION
# ------------------------------------------------------------

expected_sizes = {
    "TRUSS": 4,
    "BEAM": 6,
    "FRAME": 6
}

for etype, ndof in expected_sizes.items():

    kg = geometric_stiffness_local(
        N=1000.0,
        L=5.0,
        etype=etype
    )

    assert kg.shape == (ndof, ndof), (
        f"{etype}: Incorrect matrix dimensions"
    )

    assert np.allclose(kg, kg.T), (
        f"{etype}: Geometric stiffness matrix is not symmetric"
    )

    assert np.all(np.isfinite(kg)), (
        f"{etype}: Matrix contains invalid values"
    )

print("All geometric stiffness matrices validated.")



All geometric stiffness matrices validated.


In [67]:
# ============================================================
# GLOBAL GEOMETRIC STIFFNESS ASSEMBLY
# ============================================================

def assemble_global_geometric_stiffness(u):

    KG_GLOBAL = np.zeros_like(K_ELASTIC)

    axial_forces = {}

    for sid, result in ELEMENT_RESULTS.items():

        dofs = np.asarray(result["dofs"], dtype=int)

        T = np.asarray(result["T"], dtype=float)

        L = float(result["L"])

        # Current axial force estimate
        N = axial_force_from_state(result, u)

        # Local geometric stiffness
        KG_LOCAL = geometric_stiffness_local(
            N, L, ETYPE
        )

        # Transform into global coordinates
        KG_GLOBAL_ELEMENT = T.T @ KG_LOCAL @ T

        # Assembly
        KG_GLOBAL[np.ix_(dofs, dofs)] += KG_GLOBAL_ELEMENT

        axial_forces[sid] = N

    if not np.allclose(
        KG_GLOBAL,
        KG_GLOBAL.T,
        atol=1e-7
    ):
        raise ValueError("Global geometric stiffness is not symmetric.")

    return KG_GLOBAL, axial_forces


# Initial zero-displacement verification

KG_ZERO, N_ZERO = assemble_global_geometric_stiffness(
    np.zeros(TOTAL_DOF)
)

assert np.allclose(KG_ZERO, 0.0)

print("Global geometric stiffness assembly successful.")

Global geometric stiffness assembly successful.


In [68]:
# ============================================================
# INCREMENTAL INITIAL-STRESS GEOMETRIC STIFFNESS SOLVER
# ============================================================

def run_geometric_stiffness_analysis(
    n_steps,
    max_iterations=100,
    displacement_tolerance=1e-8,
    residual_tolerance=1e-7,
    relaxation=0.5
):

    if n_steps < 1:
        raise ValueError("Number of load steps must be positive.")

    if not 0 < relaxation <= 1:
        raise ValueError("Relaxation must be in (0,1].")

    u = np.zeros(TOTAL_DOF)

    history = []

    all_converged = True

    final_KG = np.zeros_like(K_ELASTIC)
    final_KT = K_ELASTIC.copy()
    final_N = {}

    for step in range(1, n_steps + 1):

        lam = step / n_steps

        F_step = lam * F_FULL

        converged = False

        for iteration in range(1, max_iterations + 1):

            # Update geometric stiffness
            KG, N = assemble_global_geometric_stiffness(u)

            # Tangent approximation
            KT = K_ELASTIC + KG

            # Reduced system
            Kr = Z.T @ KT @ Z

            Fr = Z.T @ F_step

            if np.linalg.matrix_rank(Kr) < Kr.shape[0]:

                raise np.linalg.LinAlgError(
                    "Reduced tangent stiffness is singular."
                )

            # Solve tangent-equilibrium approximation
            q_target = np.linalg.solve(Kr, Fr)

            u_target = Z @ q_target

            # Under-relaxed update
            du = relaxation * (u_target - u)

            u_new = u + du

            # Reassemble at updated displacement
            KG_new, N_new = assemble_global_geometric_stiffness(u_new)

            KT_new = K_ELASTIC + KG_new

            # Residual of this initial-stress approximation
            residual = F_step - KT_new @ u_new

            residual_norm = np.linalg.norm(Z.T @ residual)

            du_norm = np.linalg.norm(du)

            scale_u = max(
                np.linalg.norm(u_new),
                1e-12
            )

            scale_F = max(
                np.linalg.norm(Z.T @ F_step),
                1.0
            )

            relative_du = du_norm / scale_u

            relative_residual = residual_norm / scale_F

            u = u_new

            if (
                relative_du < displacement_tolerance
                and relative_residual < residual_tolerance
            ):

                converged = True

                break

        history.append({

            "step": step,
            "load_factor": lam,
            "iterations": iteration,
            "converged": converged,
            "relative_displacement_change": relative_du,
            "relative_residual": relative_residual

        })

        print(
            f"Step {step}/{n_steps} | "
            f"Iteration {iteration} | "
            f"du={relative_du:.3e} | "
            f"Residual={relative_residual:.3e} | "
            f"Converged={converged}"
        )

        if not converged:

            all_converged = False

            print("Load step failed to converge.")

            break

    final_KG, final_N = assemble_global_geometric_stiffness(u)

    final_KT = K_ELASTIC + final_KG

    return {

        "displacements": u,
        "K_G": final_KG,
        "K_T": final_KT,
        "axial_forces": final_N,
        "history": history,
        "all_converged": all_converged

    }


RESULT = run_geometric_stiffness_analysis(
    n_steps=n_steps,
    max_iterations=max_iterations,
    displacement_tolerance=displacement_tolerance,
    residual_tolerance=residual_tolerance,
    relaxation=relaxation
)

D_FINAL = RESULT["displacements"]

K_G_FINAL = RESULT["K_G"]

K_T_FINAL = RESULT["K_T"]

AXIAL_FORCES_FINAL = RESULT["axial_forces"]

Step 1/5 | Iteration 4 | du=2.900e-12 | Residual=5.304e-16 | Converged=True
Step 2/5 | Iteration 4 | du=1.160e-11 | Residual=2.458e-15 | Converged=True
Step 3/5 | Iteration 4 | du=2.613e-11 | Residual=8.076e-15 | Converged=True
Step 4/5 | Iteration 4 | du=4.648e-11 | Residual=1.939e-14 | Converged=True
Step 5/5 | Iteration 4 | du=7.268e-11 | Residual=3.791e-14 | Converged=True


In [75]:
# ============================================================
# CHECK SOLVER CONVERGENCE
# ============================================================

if not RESULT["all_converged"]:
    raise RuntimeError(
        "Geometric stiffness analysis did not converge. "
        "Do not treat the current displacements as final results."
    )

D_FINAL = RESULT["displacements"]
K_G_FINAL = RESULT["K_G"]
K_T_FINAL = RESULT["K_T"]
AXIAL_FORCES_FINAL = RESULT["axial_forces"]

print("Geometric stiffness analysis completed.")

Geometric stiffness analysis completed.


In [76]:
# ============================================================
# GENERAL 2D MEMBER FORCE RECOVERY
# ============================================================

def recover_segment_end_forces(u, load_factor=1.0):

    forces = {}

    for sid, result in ELEMENT_RESULTS.items():

        # Extract element displacement data
        dofs, T, ug, ul = local_displacement(result, u)

        k_local = np.asarray(
            result["k_local"], dtype=float
        )

        expected = k_local.shape[0]

        # Validate element dimensions
        if k_local.shape != (expected, expected):
            raise ValueError(
                f"Segment {sid}: invalid local stiffness matrix."
            )

        if ul.size != expected:
            raise ValueError(
                f"Segment {sid}: displacement dimension mismatch."
            )

        # Retrieve equivalent local element load
        f_load = np.asarray(
            ELEMENT_LOAD_VECTORS.get(
                sid,
                np.zeros(expected)
            ),
            dtype=float
        ).reshape(-1)

        if f_load.size != expected:
            raise ValueError(
                f"Segment {sid}: invalid local load vector."
            )

        # Elastic local end-force recovery
        f_local = k_local @ ul - load_factor * f_load

        # Axial force
        N = axial_force_from_state(result, u)

        forces[sid] = {
            "original_eid": result.get("original_eid", sid),
            "x_start": result.get("x_start", 0.0),
            "x_end": result.get("x_end", result["L"]),
            "local_displacement": ul,
            "local_end_force": f_local,
            "axial_force": N
        }

    return forces


# ============================================================
# RECOVER FINAL MEMBER FORCES
# ============================================================

if not RESULT["all_converged"]:
    raise RuntimeError(
        "Cannot recover final forces: solver did not converge."
    )

final_load_factor = RESULT["history"][-1]["load_factor"]

SEGMENT_FORCES = recover_segment_end_forces(
    D_FINAL,
    final_load_factor
)


# ============================================================
# DISPLAY NODAL DISPLACEMENTS
# ============================================================

print("\nNODAL DISPLACEMENTS")
print("=" * 60)

for i, node_id in enumerate(NODE_IDS):

    dofs = range(
        i * DOF_PER_NODE,
        (i + 1) * DOF_PER_NODE
    )

    values = D_FINAL[list(dofs)]

    print(f"\nNode {node_id}")
    print(f"UX = {values[0]:.8e} m")
    print(f"UY = {values[1]:.8e} m")

    if DOF_PER_NODE >= 3:
        print(f"RZ = {values[2]:.8e} rad")


# ============================================================
# DISPLAY MEMBER AXIAL FORCES
# ============================================================

print("\nMEMBER AXIAL FORCES")
print("=" * 60)

for sid, item in SEGMENT_FORCES.items():

    print(
        f"Element {sid}: "
        f"N = {item['axial_force']:.6f} N"
    )


NODAL DISPLACEMENTS

Node 1
UX = 0.00000000e+00 m
UY = 0.00000000e+00 m

Node 2
UX = 0.00000000e+00 m
UY = 0.00000000e+00 m

Node 3
UX = 2.69406505e-03 m
UY = -5.98202609e-19 m

Node 4
UX = 3.64630427e-03 m
UY = -2.79888264e-03 m

MEMBER AXIAL FORCES
Element 1: N = 0.000000 N
Element 2: N = -5388.130093 N
Element 3: N = 3808.956893 N
Element 4: N = 1694.843251 N
Element 5: N = -0.000000 N
Element 6: N = -11195.530577 N


In [77]:
# ============================================================
# SUPPORT REACTIONS AND DIAGNOSTICS
# ============================================================

F_final = final_load_factor * F_FULL

# Residual of the final tangent-equilibrium approximation
residual_full = K_T_FINAL @ D_FINAL - F_final

# Reactions must lie in the constraint-force space
if C.shape[0] > 0:

    multipliers, *_ = np.linalg.lstsq(
        C.T,
        residual_full,
        rcond=None
    )

    R_support = C.T @ multipliers

else:

    multipliers = np.array([])

    R_support = np.zeros(TOTAL_DOF)


free_residual = Z.T @ residual_full

print("=" * 60)
print("NONLINEAR ANALYSIS DIAGNOSTICS")
print("=" * 60)

print("All load steps converged:", RESULT["all_converged"])

print("Constraint violation:",
      np.linalg.norm(C @ D_FINAL))

print("Free residual norm:",
      np.linalg.norm(free_residual))

print("Geometric stiffness symmetry error:",
      np.linalg.norm(K_G_FINAL - K_G_FINAL.T))

print("\nSUPPORT REACTIONS")

for i, value in enumerate(R_support):

    if abs(value) > 1e-8:

        print(f"DOF {i+1}: {value:.6f} N")


print("\nCONVERGENCE HISTORY")

for row in RESULT["history"]:

    print(row)


# Save results

output = {

    "ETYPE": ETYPE,

    "load_steps": n_steps,

    "displacements": D_FINAL,

    "K_G": K_G_FINAL,

    "K_T": K_T_FINAL,

    "axial_forces": AXIAL_FORCES_FINAL,

    "segment_forces": SEGMENT_FORCES,

    "reaction_vector": R_support,

    "history": RESULT["history"],

    "all_converged": RESULT["all_converged"],

    "method_note": (
        "Incremental initial-stress fixed-point analysis; "
        "not a consistent Newton-Raphson formulation."
    )

}

with open("geometric_results.pkl", "wb") as f:

    pickle.dump(output, f)

print("\nResults saved successfully.")

NONLINEAR ANALYSIS DIAGNOSTICS
All load steps converged: True
Constraint violation: 2.6827479531585956e-19
Free residual norm: 4.2385692978446066e-10
Geometric stiffness symmetry error: 0.0

SUPPORT REACTIONS
DOF 1: -1199.207570 N
DOF 2: -1197.662742 N
DOF 3: -3800.792430 N
DOF 4: 15006.540338 N
DOF 6: -3808.877596 N

CONVERGENCE HISTORY
{'step': 1, 'load_factor': 0.2, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(2.899788739115339e-12), 'relative_residual': np.float64(5.304318013384658e-16)}
{'step': 2, 'load_factor': 0.4, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(1.1604964841864548e-11), 'relative_residual': np.float64(2.4575526537705356e-15)}
{'step': 3, 'load_factor': 0.6, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(2.6128940354675773e-11), 'relative_residual': np.float64(8.076439892449408e-15)}
{'step': 4, 'load_factor': 0.8, 'iterations': 4, 'converged': True, 'relative_displa

In [78]:
# ============================================================
# GLOBAL FORCE AND MOMENT EQUILIBRIUM
# ============================================================

F_ext = np.asarray(F_final).reshape(-1)

R = np.asarray(R_support).reshape(-1)

nodes = FEM_MODEL["nodes"]

if isinstance(nodes, dict):

    node_coords = {

        node_id: np.asarray(coord, dtype=float)[:2]

        for node_id, coord in nodes.items()

    }

else:

    nodes_array = np.asarray(nodes, dtype=float)

    node_coords = {

        node_id: nodes_array[i, :2]

        for i, node_id in enumerate(NODE_IDS)

    }


node_index = {

    node_id: i

    for i, node_id in enumerate(NODE_IDS)

}


def node_dofs(node_id):

    i = node_index[node_id]

    start = i * DOF_PER_NODE

    return list(range(start, start + DOF_PER_NODE))


sum_Fx = 0.0

sum_Fy = 0.0

sum_Mz = 0.0


for node_id in NODE_IDS:

    dofs = node_dofs(node_id)

    x, y = node_coords[node_id]

    Fx = F_ext[dofs[0]] + R[dofs[0]]

    Fy = F_ext[dofs[1]] + R[dofs[1]]

    sum_Fx += Fx

    sum_Fy += Fy

    sum_Mz += x * Fy - y * Fx

    # Include nodal moments for beam/frame structures
    if DOF_PER_NODE >= 3:

        sum_Mz += F_ext[dofs[2]] + R[dofs[2]]


print("=" * 60)

print("GLOBAL EQUILIBRIUM CHECK")

print("=" * 60)

print(f"Sum Fx = {sum_Fx:.8e} N")

print(f"Sum Fy = {sum_Fy:.8e} N")

print(f"Sum Mz = {sum_Mz:.8e} N.m")

print("\nFree DOF residual:")

print(np.linalg.norm(Z.T @ residual_full))

GLOBAL EQUILIBRIUM CHECK
Sum Fx = 1.81898940e-11 N
Sum Fy = -4.00177669e-10 N
Sum Mz = 3.27016919e+01 N.m

Free DOF residual:
4.2385692978446066e-10


In [79]:
# Reactions for the current tangent-equilibrium approximation.
F_final = final_load_factor * F_FULL
residual_full = K_T_FINAL @ D_FINAL - F_final

# Solve residual_full = C.T @ lambda for generalized support reactions.
if C.shape[0]:
    lambda_reactions, *_ = np.linalg.lstsq(C.T, residual_full, rcond=None)
    reaction_vector = C.T @ lambda_reactions
else:
    lambda_reactions = np.array([])
    reaction_vector = np.zeros(TOTAL_DOF)

print("CONVERGENCE SUMMARY")
for row in RESULT["history"]:
    print(row)
print("\nDIAGNOSTICS")
print("All load steps converged:", RESULT["all_converged"])
print("||C @ u||:", np.linalg.norm(C @ D_FINAL))
print("||Z.T @ residual||:", np.linalg.norm(Z.T @ residual_full))
print("K_G symmetry error:", np.linalg.norm(K_G_FINAL - K_G_FINAL.T))
print("Reaction multipliers:", lambda_reactions)
print("Reaction vector:", reaction_vector)

# Save results for subsequent plotting/postprocessing.
output = {
    "ETYPE": ETYPE,
    "load_steps": n_steps,
    "load_increment": load_increment,
    "displacements": D_FINAL,
    "K_G": K_G_FINAL,
    "K_T": K_T_FINAL,
    "axial_forces": AXIAL_FORCES_FINAL,
    "segment_forces": SEGMENT_FORCES,
    "reaction_multipliers": lambda_reactions,
    "reaction_vector": reaction_vector,
    "history": RESULT["history"],
    "all_converged": RESULT["all_converged"],
    "method_note": "Incremental initial-stress / P-Delta iteration; not a consistent Newton-Raphson formulation."
}
with open("geometric_results.pkl", "wb") as f:
    pickle.dump(output, f)
print("\nSaved results to geometric_results.pkl")

CONVERGENCE SUMMARY
{'step': 1, 'load_factor': 0.2, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(2.899788739115339e-12), 'relative_residual': np.float64(5.304318013384658e-16)}
{'step': 2, 'load_factor': 0.4, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(1.1604964841864548e-11), 'relative_residual': np.float64(2.4575526537705356e-15)}
{'step': 3, 'load_factor': 0.6, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(2.6128940354675773e-11), 'relative_residual': np.float64(8.076439892449408e-15)}
{'step': 4, 'load_factor': 0.8, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(4.648225396801375e-11), 'relative_residual': np.float64(1.939038923286791e-14)}
{'step': 5, 'load_factor': 1.0, 'iterations': 4, 'converged': True, 'relative_displacement_change': np.float64(7.26771317794858e-11), 'relative_residual': np.float64(3.7910916309296374e-14)}

DIAGNOSTICS
All load 

In [81]:
import numpy as np

print("=" * 60)
print("FINAL TOTAL TANGENT STIFFNESS MATRIX")
print("=" * 60)

print(np.array2string(
    K_T_FINAL,
    precision=4,
    suppress_small=True,
    max_line_width=180
))

FINAL TOTAL TANGENT STIFFNESS MATRIX
[[ 5414333.4059  1414093.7189 -4000000.            0.            0.            0.     -1414333.4059 -1414093.7189]
 [ 1414093.7189  5414333.4059        0.            0.            0.     -4000000.     -1414093.7189 -1414333.4059]
 [-4000000.            0.      5411593.4579 -1414594.5607 -1413832.564   1414594.5607     2239.1061        0.    ]
 [       0.            0.     -1414594.5607  5413832.564   1414594.5607 -1413832.564         0.     -4000000.    ]
 [       0.            0.     -1413832.564   1414594.5607  5413832.564  -1414594.5607 -4000000.            0.    ]
 [       0.     -4000000.      1414594.5607 -1413832.564  -1414594.5607  5414594.3554        0.         -761.7914]
 [-1414333.4059 -1414093.7189     2239.1061        0.     -4000000.            0.      5412094.2998  1414093.7189]
 [-1414093.7189 -1414333.4059        0.     -4000000.            0.         -761.7914  1414093.7189  5415095.1973]]
